# Guilliams dataset inspection

Inspect GSE192742 inputs, prepare sample metadata, and evaluate RNA-only QC for human library CS43. This notebook does not perform final filtering, doublet detection, cell-type annotation, or cross-species integration.

## 1. Setup and archive inventory

Dependencies: numpy, pandas, and h5py. Install missing packages with `%pip install numpy pandas h5py` in the selected notebook environment.

Run from the repository root or its pipeline directory. Set `GUILLIAMS_PROJECT_DIR` if the raw archive is stored elsewhere.


In [ ]:
import os
import csv
import gzip
import io
import re
import time
import tarfile
from pathlib import Path
from collections import Counter
from urllib.request import Request, urlopen

import h5py
import numpy as np
import pandas as pd

import os

# Locate the shared project folder from the notebook working directory.
_start = Path.cwd().resolve()
_override = os.environ.get("GUILLIAMS_PROJECT_DIR")
if _override:
    _project_root = Path(_override).expanduser().resolve()
else:
    _project_root = next(
        (p for p in (_start, *_start.parents)
         if (p / "guilliams_metadata").is_dir()),
        None,
    )
    if _project_root is None:
        raise FileNotFoundError(
            "Keep guilliams_metadata beside the repository, "
            "or set GUILLIAMS_PROJECT_DIR."
        )
PROJECT_DIR = Path(os.path.relpath(_project_root, _start))
project_dir = PROJECT_DIR
data_path = project_dir / "GSE192742_RAW.tar"
archive_path = data_path
metadata_dir = project_dir / "guilliams_metadata"
qc_dir = project_dir / "guilliams_qc"
if not data_path.is_file():
    raise FileNotFoundError(f"Archive not found: {data_path}. Set GUILLIAMS_PROJECT_DIR.")


In [ ]:
print("File exists:", data_path.exists())

with tarfile.open(data_path, "r") as archive:
    filenames = archive.getnames()

print("Total files:", len(filenames))
print("\nFirst 10 files:")
for name in filenames[:10]:
    print(name)


In [ ]:
# Read only the first H5 file without extracting the entire archive.
sample_file = next(name for name in filenames if name.endswith(".h5"))
print("Selected file:", sample_file)

with tarfile.open(data_path, "r") as archive:
    with archive.extractfile(sample_file) as source:
        content = io.BytesIO(source.read())

with h5py.File(content, "r") as f:
    print("Top-level groups:", list(f.keys()))

    def show_dataset(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(name, "Shape:", obj.shape)

    f.visititems(show_dataset)


## 2. GEO metadata and sample selection

Reuse cached GEO records where available. Condition assignments follow Crystal's donor mapping, not an inferred clinical diagnosis. Unlisted donors remain unknown; all 43 human libraries remain in the review manifest.


In [ ]:
# Collect unique GEO sample IDs from the archive filenames.
sample_ids = sorted({
    match.group(0)
    for name in filenames
    if (match := re.match(r"GSM\d+", name))
})

metadata_dir = data_path.parent / "guilliams_metadata"
metadata_dir.mkdir(parents=True, exist_ok=True)

print("Total samples:", len(sample_ids))
failed_samples = []

for index, accession in enumerate(sample_ids, start=1):
    output_file = metadata_dir / f"{accession}.txt"

    # Reuse previously downloaded metadata when valid.
    if output_file.exists():
        cached_text = output_file.read_text(encoding="utf-8")
        if f"!Sample_geo_accession = {accession}" in cached_text:
            continue

    url = (
        "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi"
        f"?acc={accession}&targ=self&form=text&view=full"
    )
    request = Request(url, headers={"User-Agent": "Mozilla/5.0"})

    try:
        with urlopen(request, timeout=60) as response:
            text = response.read().decode("utf-8")

        if f"!Sample_geo_accession = {accession}" not in text:
            raise ValueError("Expected GEO metadata was not returned.")

        output_file.write_text(text, encoding="utf-8")
        print(f"[{index}/{len(sample_ids)}] Saved {accession}")

    except Exception as error:
        failed_samples.append(accession)
        print(f"[{index}/{len(sample_ids)}] Failed {accession}: {error}")

    # Pause between requests to avoid overloading GEO.
    time.sleep(1)

print("\nDownload pass complete.")
print("Failed samples:", failed_samples)

if failed_samples:
    raise RuntimeError(f"Metadata download incomplete: {failed_samples}")


In [ ]:
# Parse GEO metadata while preserving all characteristic fields.
sample_rows = []

for accession in sample_ids:
    text = (metadata_dir / f"{accession}.txt").read_text(encoding="utf-8")
    row = {"accession": accession}

    field_names = {
        "!Sample_title": "title",
        "!Sample_source_name_ch1": "tissue",
        "!Sample_organism_ch1": "organism",
    }

    for line in text.splitlines():
        key, separator, value = line.partition(" = ")
        if not separator:
            continue

        if key in field_names:
            row[field_names[key]] = value.strip()

        elif key == "!Sample_characteristics_ch1":
            label, colon, detail = value.partition(":")
            if colon:
                column = "characteristic_" + label.strip()
                detail = detail.strip()
                row[column] = (
                    row[column] + " | " + detail
                    if column in row else detail
                )

    sample_rows.append(row)

# Save a sample manifest without assigning inferred disease or donor labels.
columns = ["accession", "title", "organism", "tissue"]
columns += sorted({
    key for row in sample_rows for key in row
    if key not in columns
})

manifest_path = metadata_dir / "sample_manifest.csv"

with manifest_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=columns)
    writer.writeheader()
    writer.writerows(sample_rows)

print("Samples:", len(sample_rows))
print("\nSamples by organism:")
for organism, count in Counter(
    row.get("organism", "Unknown") for row in sample_rows
).items():
    print(f"{organism}: {count}")

print("\nSamples by digestion method:")
for method, count in Counter(
    row.get("characteristic_digestion method", "Unknown")
    for row in sample_rows
).items():
    print(f"{method}: {count}")

print("\nAvailable columns:")
print("\n".join(columns))
print("\nManifest saved to:", manifest_path)


In [ ]:
# Reload the saved manifest so this cell works after a kernel restart.
manifest_path = metadata_dir / "sample_manifest.csv"

with manifest_path.open(encoding="utf-8", newline="") as handle:
    sample_rows = list(csv.DictReader(handle))

human_rows = [
    row for row in sample_rows
    if row.get("organism") == "Homo sapiens"
]

print("Human samples:", len(human_rows))
display(pd.DataFrame(human_rows)[["accession", "title", "organism"]].head(6))


In [ ]:
healthy_donors = {
    "H06", "H07", "H10", "H13", "H16",
    "H18", "H22", "H23", "H25",
}
disease_donors = {"H04", "H11", "H21"}

# Extract explicit H identifiers from titles.
# Leave missing or ambiguous donor IDs unresolved.
human_review = []

for row in human_rows:
    title = row.get("title", "")
    matches = re.findall(
        r"(?<![A-Za-z0-9])H(\d+)(?![A-Za-z0-9])",
        title,
    )
    donors = sorted({f"H{int(number):02d}" for number in matches})
    donor = donors[0] if len(donors) == 1 else "unresolved"

    if donor in healthy_donors:
        condition = "healthy"
    elif donor in disease_donors:
        condition = "disease"
    else:
        condition = "unknown"

    human_review.append({
        **row,
        "donor_from_title": donor,
        "team_condition": condition,
    })

print("\nSample counts by team condition:")
print(dict(Counter(row["team_condition"] for row in human_review)))

print("\nDonors requiring review:")
print(sorted({
    row["donor_from_title"]
    for row in human_review
    if row["team_condition"] == "unknown"
}))


In [ ]:
archive_path = project_dir / "GSE192742_RAW.tar"
output_path = project_dir / "guilliams_metadata/human_sample_review.csv"

# Identify spatial samples from their coordinate files.
with tarfile.open(archive_path, "r") as archive:
    spatial_ids = {
        Path(name).name.split("_")[0]
        for name in archive.getnames()
        if "tissue_positions" in name
    }

review_rows = []

for row in human_review:
    accession = row["accession"]
    method = row.get("characteristic_digestion method", "")

    if accession in spatial_ids:
        assay_group = "spatial"
    elif method == "Nuclei":
        assay_group = "single_nucleus"
    elif method in {"Ex Vivo", "In Vivo"}:
        assay_group = "single_cell"
    else:
        assay_group = "unresolved"

    review_rows.append({
        "accession": accession,
        "sample_name": row.get("characteristic_shortfilename", ""),
        "donor": row["donor_from_title"],
        "donor_source": "GEO sample title",
        "condition": row["team_condition"],
        "condition_source": (
            "Crystal-provided donor mapping"
            if row["team_condition"] != "unknown"
            else "Not assigned"
        ),
        "assay_group": assay_group,
        "digestion_method": method,
        "original_condition": row.get("characteristic_condition", ""),
        "original_title": row["title"],
    })

with output_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(review_rows[0]))
    writer.writeheader()
    writer.writerows(review_rows)

print("Saved:", output_path)
print("Total samples:", len(review_rows))

print("\nSamples by assay group and condition:")
counts = Counter(
    (row["assay_group"], row["condition"])
    for row in review_rows
)
for (assay, condition), count in sorted(counts.items()):
    print(f"{assay} | {condition}: {count}")

print("\nUnique donors by condition:")
for condition in ["healthy", "disease", "unknown"]:
    donors = sorted({
        row["donor"]
        for row in review_rows
        if row["condition"] == condition
        and row["donor"] != "unresolved"
    })
    print(f"{condition}: {len(donors)} donors — {', '.join(donors)}")


## 3. Matrix format inspection

Inspect the 20 labeled human single-cell libraries first. This is a working subset, not the full third dataset. Other human libraries and mouse preprocessing remain outside this inspection.


In [ ]:
review_path = project_dir / "guilliams_metadata/human_sample_review.csv"
archive_path = project_dir / "GSE192742_RAW.tar"

with review_path.open(encoding="utf-8", newline="") as handle:
    review_rows = list(csv.DictReader(handle))

# Review the labeled single-cell subset first.
# Other samples remain unchanged in the full manifest.
labeled_samples = [
    row for row in review_rows
    if row["assay_group"] == "single_cell"
    and row["condition"] in {"healthy", "disease"}
]

with tarfile.open(archive_path, "r") as archive:
    matrix_files = [
        member for member in archive.getmembers()
        if member.isfile()
        and "filtered_feature_bc_matrix" in member.name
        and member.name.endswith((".h5", ".txt.gz"))
    ]

format_counts = Counter()
issues = []

print("Accession | Donor | Condition | Matrix file | Size (MB)")

for row in labeled_samples:
    matches = [
        member for member in matrix_files
        if Path(member.name).name.split("_")[0] == row["accession"]
    ]

    if len(matches) != 1:
        issues.append((row["accession"], len(matches)))

    for member in matches:
        file_format = "h5" if member.name.endswith(".h5") else "txt.gz"
        format_counts[file_format] += 1

        print(
            row["accession"],
            row["donor"],
            row["condition"],
            member.name,
            f"{member.size / 1024**2:.1f}",
            sep=" | ",
        )

print("\nMatrix formats:", dict(format_counts))
print("Samples without exactly one matrix:", issues)


In [ ]:
target_file = "GSM5764368_filtered_feature_bc_matrix_CS43.txt.gz"


# Read the header and five gene rows only.
with tarfile.open(archive_path, "r") as archive:
    source = archive.extractfile(target_file)
    if source is None:
        raise ValueError("Selected file could not be read.")

    with source:
        with gzip.open(source, "rt", encoding="utf-8") as handle:
            reader = csv.reader(handle, delimiter="\t")
            header = next(reader)
            preview_rows = [next(reader) for _ in range(5)]

# Handle an optional empty field above the gene-name column.
barcodes = header[1:] if header[0] == "" else header

print("Number of cell columns:", len(barcodes))
print("Fields in the first gene row:", len(preview_rows[0]))

for row in preview_rows:
    assert len(row) == len(barcodes) + 1, "Unexpected row width."

preview = pd.DataFrame(
    [[int(value) for value in row[1:6]] for row in preview_rows],
    index=[row[0] for row in preview_rows],
    columns=barcodes[:5],
)
preview.index.name = "gene"

display(preview)


In [ ]:
gene_names = []
total_counts = np.zeros(len(barcodes), dtype=np.int64)
detected_features = np.zeros(len(barcodes), dtype=np.int64)

with tarfile.open(archive_path, "r") as archive:
    source = archive.extractfile(target_file)
    if source is None:
        raise ValueError("Selected file could not be read.")

    with source:
        with gzip.open(source, "rt", encoding="utf-8") as handle:
            reader = csv.reader(handle, delimiter="\t")
            next(reader)  # Skip the barcode header.

            for line_number, row in enumerate(reader, start=2):
                if len(row) != len(barcodes) + 1:
                    raise ValueError(f"Unexpected row width at line {line_number}")

                values = np.asarray(row[1:], dtype=np.float64)

                if (
                    not np.isfinite(values).all()
                    or (values < 0).any()
                    or (values != np.floor(values)).any()
                ):
                    raise ValueError(f"Invalid counts at line {line_number}")

                counts = values.astype(np.int64)
                gene_names.append(row[0])
                total_counts += counts
                detected_features += counts > 0

name_counts = Counter(gene_names)
duplicate_names = {
    name: count for name, count in name_counts.items() if count > 1
}

print("File:", target_file)
print("Feature rows:", len(gene_names))
print("Cell columns:", len(barcodes))
print("Duplicate barcodes:", len(barcodes) - len(set(barcodes)))
print("Repeated feature names:", len(duplicate_names))
print("Zero-count cells:", int((total_counts == 0).sum()))
print("Median total counts:", float(np.median(total_counts)))
print("Median detected feature rows:", float(np.median(detected_features)))

print("\nFirst 10 feature names:")
print(gene_names[:10])

print("\nLast 20 feature names:")
print(gene_names[-20:])


## 4. RNA feature alignment

CS43 includes additional features consistent with antibody measurements. Compare its leading rows with the gene-only CS41 H5 reference before computing RNA QC. Name matching below is a compatibility check, not an exact reconstruction of the export software's naming algorithm. This evidence applies to CS43 only.


In [ ]:
# Inspect feature types in another library from donor H06.
reference_file = "GSM5764363_filtered_feature_bc_matrix_CS41.h5"

with tarfile.open(archive_path, "r") as archive:
    source = archive.extractfile(reference_file)
    if source is None:
        raise ValueError("Reference file could not be read.")

    with source:
        reference_content = io.BytesIO(source.read())

with h5py.File(reference_content, "r") as f:
    features = f["matrix/features"]

    reference_names = features["name"].asstr()[:]
    reference_types = features["feature_type"].asstr()[:]
    reference_ids = features["id"].asstr()[:]

print("Reference file:", reference_file)
print("Feature types:", dict(Counter(reference_types)))

for feature_type in sorted(set(reference_types)):
    positions = [
        i for i, value in enumerate(reference_types)
        if value == feature_type
    ]

    print(f"\n{feature_type}: {len(positions)} features")
    print("First 5:")
    for i in positions[:5]:
        print(reference_ids[i], reference_names[i], sep="\t")

    print("Last 5:")
    for i in positions[-5:]:
        print(reference_ids[i], reference_names[i], sep="\t")


In [ ]:
# Compare the text matrix's leading rows with the H5 gene reference.
rna_reference = [
    str(name)
    for name, feature_type in zip(reference_names, reference_types)
    if feature_type == "Gene Expression"
]

n_reference = len(rna_reference)
leading_names = gene_names[:n_reference]

# This is a diagnostic comparison only; original names are unchanged.
mismatches = [
    (i + 1, text_name, reference_name)
    for i, (text_name, reference_name) in enumerate(
        zip(leading_names, rna_reference)
    )
    if text_name != reference_name.replace("-", ".")
]

print("Reference gene rows:", n_reference)
print("Text feature rows:", len(gene_names))
print("Text has enough rows:", len(gene_names) >= n_reference)
print("Name mismatches after hyphen-to-dot comparison:", len(mismatches))

print("\nFirst 15 mismatches (row, text name, reference name):")
for item in mismatches[:15]:
    print(item)

print("\nFeatures around the proposed boundary:")
for i in range(
    max(0, n_reference - 5),
    min(len(gene_names), n_reference + 10),
):
    print(i + 1, gene_names[i], sep="\t")

print("\nAdditional rows:", max(0, len(gene_names) - n_reference))


In [ ]:
# Test naming conventions without changing the original feature names.
def candidate_export_name(name):
    name = re.sub(r"[^A-Za-z0-9._]", ".", name)

    if re.match(r"^[0-9]|^\.[0-9]", name):
        name = "X" + name

    return name

expected_names = [
    candidate_export_name(name)
    for name in rna_reference
]
expected_counts = Counter(expected_names)

unresolved = []
suffix_matches = []

for row_number, (actual, expected) in enumerate(
    zip(gene_names[:n_reference], expected_names),
    start=1,
):
    if actual == expected:
        continue

    suffix = re.fullmatch(re.escape(expected) + r"\.(\d+)", actual)

    # A numeric suffix is plausible only if the base name is repeated.
    if suffix and expected_counts[expected] > 1:
        suffix_matches.append((row_number, actual, expected))
    else:
        unresolved.append((row_number, actual, expected))

print("Compared rows:", min(len(gene_names), n_reference))
print("Plausible duplicate-name suffixes:", len(suffix_matches))
print("Unresolved differences:", len(unresolved))

for item in unresolved[:30]:
    print(item)


## 5. Preliminary RNA QC for CS43

Evaluate Crystal's existing human-pipeline cutoffs (at least 300 and fewer than 6,000 detected gene rows; mitochondrial counts below 30%). These are provisional for Guilliams. No cells are removed and doublets have not been assessed. Counts of detected rows are not counts of unique gene symbols.


In [ ]:
if len(gene_names) < n_reference or unresolved:
    raise ValueError("RNA feature alignment must pass before QC.")
if len(set(barcodes)) != len(barcodes):
    raise ValueError("Duplicate cell barcodes.")

# Compute RNA-only QC for CS43 without filtering any cells.
rna_counts = np.zeros(len(barcodes), dtype=np.int64)
rna_detected = np.zeros(len(barcodes), dtype=np.int64)
mitochondrial_counts = np.zeros(len(barcodes), dtype=np.int64)

# Use the original H5 gene symbols to identify mitochondrial genes.
mitochondrial_mask = np.array([
    name.upper().startswith("MT-")
    for name in rna_reference
])

with tarfile.open(archive_path, "r") as archive:
    source = archive.extractfile(target_file)
    if source is None:
        raise ValueError("Selected file could not be read.")

    with source:
        with gzip.open(source, "rt", encoding="utf-8") as handle:
            reader = csv.reader(handle, delimiter="\t")
            next(reader)

            for index in range(n_reference):
                row = next(reader)

                if row[0] != gene_names[index]:
                    raise ValueError(f"Feature order changed at row {index + 1}")

                counts = np.asarray(row[1:], dtype=np.int64)
                rna_counts += counts
                rna_detected += counts > 0

                if mitochondrial_mask[index]:
                    mitochondrial_counts += counts

# Mitochondrial percentage is undefined for cells with zero RNA counts.
pct_mt = np.divide(
    mitochondrial_counts * 100.0,
    rna_counts,
    out=np.full(len(barcodes), np.nan),
    where=rna_counts > 0,
)

qc_preview = pd.DataFrame({
    "total_rna_counts": rna_counts,
    "detected_gene_rows": rna_detected,
    "pct_counts_mt": pct_mt,
})

print("File:", target_file)
print("RNA feature rows:", n_reference)
print("Mitochondrial gene rows:", int(mitochondrial_mask.sum()))
print("Zero-RNA-count cells:", int((rna_counts == 0).sum()))

display(
    qc_preview.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(2)
)


In [ ]:
# Check the current QC cutoffs before removing any cells.
low_genes = qc_preview["detected_gene_rows"] < 300
high_genes = qc_preview["detected_gene_rows"] >= 6000
high_mt = qc_preview["pct_counts_mt"] >= 30
invalid_mt = qc_preview["pct_counts_mt"].isna()

flagged = low_genes | high_genes | high_mt | invalid_mt

print("Sample: CS43")
print("Cells before filtering:", len(qc_preview))
print("Cells with fewer than 300 genes:", int(low_genes.sum()))
print("Cells with 6000 or more genes:", int(high_genes.sum()))
print("Cells with mt percent >= 30:", int(high_mt.sum()))
print("Cells with missing mt percent:", int(invalid_mt.sum()))
print("Cells flagged:", int(flagged.sum()))
print("Cells kept:", int((~flagged).sum()))
print(f"Percent kept: {(~flagged).mean() * 100:.2f}%")


In [ ]:
# Save the per-cell QC table and the provisional flags.
qc_dir.mkdir(parents=True, exist_ok=True)

qc_export = qc_preview.copy()
qc_export.insert(0, "barcode", barcodes)
qc_export.insert(0, "sample", "GSM5764368")
qc_export.insert(0, "donor", "H06")

qc_export["flag_low_genes"] = low_genes.to_numpy()
qc_export["flag_high_genes"] = high_genes.to_numpy()
qc_export["flag_high_mt"] = high_mt.to_numpy()
qc_export["flag_invalid_mt"] = invalid_mt.to_numpy()
qc_export["passes_qc"] = (~flagged).to_numpy()

output_file = qc_dir / "GSM5764368_rna_qc.csv.gz"
qc_export.to_csv(output_file, index=False, compression="gzip")

print("Saved:", output_file)
print("Rows written:", len(qc_export))
print("No cells were removed in this preview.")


## Findings and next steps

This looks like a mixed H5/text dataset. The RNA and antibody data seem to split around CS43. I would check each text matrix separately before running the full batch. Keep the original feature names, review the unknown donor labels, and inspect sample-specific QC before final filtering.
